# Paper 22 neck on YOLO26n: CFA and C3-PD (backbone = proposed ADown + ECA)

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Proposed (fixed for the whole neck study):** ADown at layers 3, 5, 7 + ECA after layers 6 (k = 3) and 8 (k = 5) |
| **Neck (layers 11–22)** | **Paper 22 (KFP-YOLO):** **CFA** at the four Concat fusion nodes (12, 15, 18, 21) and/or **C3-PD** instead of the four neck C3k2 (13, 16, 19, 22) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

Two variants are trained:

| Run | Neck |
|---|---|
| `nk_cfa` | CFA at layers 12, 15, 18, 21 (Paper 22's best pair in its ablation: ADown + CFA) |
| `nk_c3pd_cfa` | C3-PD at layers 13, 16, 19, 22 + CFA (the full Paper 22 neck) |

**Not taken from Paper 22 here:** ADown at layer 1 (the proposed backbone keeps layer 1 as a Conv; ADown is at 3, 5, 7).

The reference row of the neck table is the **proposed backbone with the YOLO26n neck** (`final_backbone.ipynb`); it is not trained here.
Same training settings as every other row (100 epochs, seed 0, MuSGD, imgsz 640, batch 16).

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**. Then **Run All**.

## 1. Setup

In [1]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)

import time
NOTEBOOK_START = time.time()  # used by the Kaggle 12 h guard in the training cell


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.137 | torch 2.11.0+cu130 | torchvision 0.26.0+cu130
GPU: Tesla T4


## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [3]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to digitalaccess-2(copy)-4 in yolo26:: 100%|██████████| 17613/17613 [00:01<00:00, 10030.99it/s]

train : 7704 images
valid : 550 images
test  : 550 images
nc = 27
✅ Dataset prepared!


## 3. Backbone (fixed): ADown (3, 5, 7) + ECA (6, 8)

The same backbone as `final_backbone.ipynb`. ADown is set in the YAML; ECA is added by the trainer after `yolo26n.pt` is loaded.

In [4]:
%%writefile yolo26n-final-backbone.yaml
# YOLO26n with the proposed backbone: stride-2 Conv at layers 3, 5, 7 -> ADown (ECA after 6, 8 is added by the trainer).
# Neck and head = YOLO26n base.
nc: 80 # overridden by the dataset
end2end: True
reg_max: 1
scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, ADown, [256]] # 3-P3/8   (Conv -> ADown)
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, ADown, [512]] # 5-P4/16  (Conv -> ADown)
  - [-1, 2, C3k2, [512, True]] # 6  (+ ECA k=3)
  - [-1, 1, ADown, [1024]] # 7-P5/32 (Conv -> ADown)
  - [-1, 2, C3k2, [1024, True]] # 8  (+ ECA k=5)
  - [-1, 1, SPPF, [1024, 5, 3, True]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, True]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, True]] # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, True]] # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 1, C3k2, [1024, True, 0.5, True]] # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)

Writing yolo26n-final-backbone.yaml


In [5]:
%%writefile eca_backbone.py
"""Proposed backbone for YOLO26n: ECA after the C3k2 at layer 6 (P4, k=3) and layer 8 (P5, k=5). ADown (3, 5, 7) is in the YAML."""
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import C3k2
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

ECA_LAYERS = {6: 3, 8: 5}  # layer index: ECA kernel size


class ECA(nn.Module):
    """Efficient Channel Attention: global avg pool -> conv across channels (kernel k) -> sigmoid -> scale. 1-D conv as 4-D Conv2d."""

    def __init__(self, k=3):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(1, 1, (k, 1), padding=(k // 2, 0), bias=False)
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.pool(x).transpose(1, 2)  # (B, C, 1, 1) -> (B, 1, C, 1)
        y = self.act(self.conv(y)).transpose(1, 2)  # back to (B, C, 1, 1)
        return x * y


class C3k2ECA(nn.Module):
    """A pretrained C3k2 block followed by ECA."""

    def __init__(self, block, k=3):
        super().__init__()
        self.block = block
        self.eca = ECA(k)

    def forward(self, x):
        return self.eca(self.block(x))


def add_eca(model, layers=ECA_LAYERS):
    """Wrap the C3k2 at the given layers with ECA (the C3k2 keeps its weights). Returns the model."""
    for i, k in layers.items():
        old = model.model[i]
        assert isinstance(old, C3k2), f"layer {i} is {type(old).__name__}, not C3k2"
        new = C3k2ECA(old, k).to(next(old.parameters()).device)
        new.i, new.f, new.type = old.i, old.f, "C3k2ECA"  # keep the routing attributes used by the model's forward
        new.np = sum(p.numel() for p in new.parameters())
        model.model[i] = new
    return model


class ECATrainer(DetectionTrainer):
    """Builds the model from the ADown YAML, loads the pretrained weights, then adds ECA after the C3k2 at layers 6 and 8."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_eca(model)
        if RANK in {-1, 0}:
            print(f"Proposed backbone: ADown at layers 3, 5, 7 + ECA after layers {list(ECA_LAYERS)} (k = {list(ECA_LAYERS.values())})")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model

Writing eca_backbone.py


In [6]:
%%writefile fix_eca_ncnn.py
"""Re-export ECA-style models to NCNN without the Segmentation fault (papers 10, 17, 19, 49).

Why: ECA blocks do pool -> transpose(1,2) -> Conv2d(1,1,(k,1)) -> transpose. PNNX turns the first transpose into a
no-op Permute, so the 1-channel conv receives C channels and NCNN crashes (Segmentation fault).

Fix: the cross-channel 1-D conv is rewritten as an exactly equivalent 1x1 Conv2d(C, C) whose weight is a banded
matrix built from the trained kernel. Same math, same outputs, no transposes -> NCNN-safe.

Run inside the unzipped bench folder (needs bb_*.pt and the *_backbone.py file):
    python fix_eca_ncnn.py            # check + export every bb_*.pt that contains ECA (replaces <name>_ncnn_model)
    python fix_eca_ncnn.py --check    # only check equivalence, no export
Models without ECA blocks are left untouched.
"""
import glob
import importlib
import shutil
import sys
from pathlib import Path

import torch
import torch.nn as nn

sys.path.insert(0, ".")
for f in glob.glob("*_backbone.py"):  # torch needs these to unpickle the custom modules
    importlib.import_module(Path(f).stem)

from ultralytics import YOLO  # noqa: E402


class ECAConv1x1(nn.Module):
    """ECA with the cross-channel conv as a banded 1x1 Conv2d(C, C). scale=True -> x * att, False -> att only."""

    def __init__(self, conv1d: nn.Conv2d, c: int, scale: bool):
        super().__init__()
        w = conv1d.weight.detach().view(-1).float()  # (k,)
        k, p = w.numel(), conv1d.padding[0]
        band = torch.zeros(c, c)
        for o in range(c):  # out[o] = sum_j w[j] * in[o + j - p]  (zero padding)
            for j in range(k):
                i = o + j - p
                if 0 <= i < c:
                    band[o, i] = w[j]
        self.scale = scale
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(c, c, 1, bias=False)
        self.conv.weight.data.copy_(band.view(c, c, 1, 1))
        self.act = nn.Sigmoid()

    def forward(self, x):
        y = self.act(self.conv(self.pool(x)))
        return x * y if self.scale else y


def is_eca(m):
    """A block whose .conv is Conv2d(1, 1, (k, 1)) with k > 1 - the transpose-based ECA pattern."""
    c = getattr(m, "conv", None)
    return (isinstance(c, nn.Conv2d) and c.in_channels == 1 and c.out_channels == 1
            and c.kernel_size[1] == 1 and c.kernel_size[0] > 1 and not isinstance(m, ECAConv1x1))


@torch.no_grad()
def patch(net):
    """Replace every ECA block. Returns (count, max abs difference old vs new)."""
    names = [n for n, m in net.named_modules() if is_eca(m)]
    if not names:
        return 0, 0.0
    chans = {}

    def hook(name):
        def f(m, inp, out):  # must return None, otherwise it would replace the block's output
            chans.setdefault(name, inp[0].shape[1])
        return f

    hooks = [net.get_submodule(n).register_forward_hook(hook(n)) for n in names]
    net(torch.zeros(1, 3, 640, 640))
    for h in hooks:
        h.remove()

    diff = 0.0
    for n in names:
        old, c = net.get_submodule(n), chans[n]
        x = torch.randn(2, c, 20, 20)
        ref = old(x)
        new = None
        for scale in (True, False):  # find which variant this block is (returns x*att or att only)
            cand = ECAConv1x1(old.conv, c, scale).eval()
            out = cand(x)
            if out.shape == ref.shape and (out - ref).abs().max().item() < 1e-4:
                new = cand
                diff = max(diff, (out - ref).abs().max().item())
                break
        if new is None:
            sys.exit(f"Block {n} ({type(old).__name__}) did not match - not exporting.")
        parent, _, attr = n.rpartition(".")
        setattr(net.get_submodule(parent) if parent else net, attr, new)
    return len(names), diff


def main():
    pts = sorted(p for p in glob.glob("bb_*.pt") if not p.endswith("_fixed.pt"))
    if not pts:
        sys.exit("No bb_*.pt found. Run this inside the unzipped bench folder.")
    for pt in pts:
        yolo = YOLO(pt, task="detect")
        net = yolo.model.float().eval()
        n, diff = patch(net)
        if n == 0:
            print(f"{pt}: no ECA blocks - left as is.")
            continue
        print(f"{pt}: replaced {n} ECA blocks, max block output difference = {diff:.2e}  -> Check OK")
        if "--check" in sys.argv:
            continue

        name = Path(pt).stem
        tmp = Path(f"{name}_fixed.pt")
        yolo.save(tmp)
        exported = Path(YOLO(str(tmp), task="detect").export(format="ncnn", imgsz=640))
        out_dir = Path(f"{name}_ncnn_model")
        if out_dir.exists():
            shutil.rmtree(out_dir)
        shutil.move(str(exported), out_dir)
        tmp.unlink(missing_ok=True)
        Path(f"{name}_fixed.torchscript").unlink(missing_ok=True)
        print(f"{pt}: Done -> {out_dir}/")
    print("All done. Now run: python pi5_bench.py")


if __name__ == "__main__":
    main()

Writing fix_eca_ncnn.py


## 4. Paper 22 neck: CFA and C3-PD

**CFA** (Paper 22, Section 2.3.4, Eq. 7–11): Concat, then **coordinate attention** on the fused map $X$:
pool along each row and each column, a shared 1×1 conv + BN + SiLU (channels reduced to $m = \max(8, C/32)$),
then two 1×1 convs + sigmoid give a row weight $a_h$ and a column weight $a_w$:
$$Z = X \times a_h \times a_w$$

**C3-PD** (Paper 22, Section 2.3.3, Eq. 5–6): two parallel 1×1 convs (a shallow branch and a deep branch), the deep branch goes through a
**PDBlock**, the two are concatenated and a 1×1 conv gives the output. PDBlock:
$$Y = \mathrm{Conv}_{1\times1}\big(\mathrm{SE}(\mathrm{PConv}(X))\big), \qquad Z = X + Y$$
PConv = 3×3 conv on the first 1/4 of the channels, the rest pass through unchanged; SE = squeeze-and-excitation (reduction 16).

**Where:**

| Layer | YOLO26n neck | `nk_cfa` | `nk_c3pd_cfa` |
|---|---|---|---|
| 12, 15, 18, 21 | Concat | **CFA** | **CFA** |
| 13, 16, 19, 22 | C3k2 (pretrained) | C3k2 (pretrained) | **C3-PD** (new, 1 PDBlock each) |

Channel counts stay the same, so no layer index changes and the head connections stay the same.

**Start values.**
- CFA: the two output convs start with zero weights and bias 3, so $a_h \approx a_w \approx 0.95$ and the pretrained neck sees almost the plain concat at the start (Paper 22 trains from scratch and does not need this).
- C3-PD replaces pretrained C3k2 blocks, so these four layers start from random weights; the last 1×1 conv of each PDBlock starts at zero (BN scale 0), so each PDBlock starts as an identity.

**Assumptions** (not stated exactly in Paper 22): C3-PD replaces all four neck C3k2 (the paper says "the corresponding C3k2 blocks"),
one PDBlock per C3-PD, PConv ratio 1/4, SE reduction 16. Paper 22 also reports fewer parameters for "+CFA" alone (2.51 M → 2.37 M) without
explaining it; here CFA only adds the attention weights.

**Export.** Row and column pooling are written as means over one axis. In eval mode CFA applies the shared conv to the row and column
descriptors separately (identical to concat → conv → split, because BN then uses fixed statistics), so the exported graph has no transposes.

In [7]:
%%writefile cfa_neck.py
"""Paper 22 (KFP-YOLO) neck for YOLO26n: CFA (Concat + coordinate attention) at the neck fusion nodes (12, 15, 18, 21)
and C3-PD (PConv + SE blocks) instead of the neck C3k2 (13, 16, 19, 22). Backbone = proposed ADown (3, 5, 7) + ECA (6, 8)."""
import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import C3k2
from ultralytics.nn.modules.conv import Concat, Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

from eca_backbone import ECA_LAYERS, add_eca

CFA_LAYERS = (12, 15, 18, 21)
C3PD_LAYERS = (13, 16, 19, 22)
CA_BIAS = 3.0  # sigmoid(3) = 0.95: CFA starts close to the plain concat


class CFA(nn.Module):
    """Concat + coordinate attention: Z = X * a_h * a_w (Paper 22, Eq. 7-11)."""

    def __init__(self, c, n=2):
        super().__init__()
        self.n = n
        m = max(8, c // 32)
        self.conv1 = nn.Conv2d(c, m, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(m)
        self.act = nn.SiLU()
        self.conv_h = nn.Conv2d(m, c, 1)
        self.conv_w = nn.Conv2d(m, c, 1)
        for cv in (self.conv_h, self.conv_w):
            nn.init.zeros_(cv.weight)
            nn.init.constant_(cv.bias, CA_BIAS)

    def forward(self, xs):
        x = torch.cat(xs, 1)
        h, w = x.shape[2], x.shape[3]
        xh = x.mean(dim=3, keepdim=True)  # (B, C, H, 1): pool along each row
        xw = x.mean(dim=2, keepdim=True)  # (B, C, 1, W): pool along each column
        if self.training:  # as in the paper: one shared conv + BN over the joined descriptors, then split
            y = self.act(self.bn1(self.conv1(torch.cat((xh, xw.permute(0, 1, 3, 2)), 2))))
            yh, yw = y[:, :, :h], y[:, :, h:].permute(0, 1, 3, 2)
        else:  # same result with fixed BN statistics, without transposes (NCNN-safe)
            yh = self.act(self.bn1(self.conv1(xh)))
            yw = self.act(self.bn1(self.conv1(xw)))
        return x * torch.sigmoid(self.conv_h(yh)) * torch.sigmoid(self.conv_w(yw))


class PDBlock(nn.Module):
    """Z = X + Conv1x1(SE(PConv(X))) (Paper 22, Eq. 5-6). PConv: 3x3 conv on the first 1/4 of the channels."""

    def __init__(self, c, n_div=4, r=16):
        super().__init__()
        self.cp = c // n_div
        self.pconv = nn.Conv2d(self.cp, self.cp, 3, 1, 1, bias=False)
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, max(8, c // r), 1), nn.ReLU(inplace=True),
                                nn.Conv2d(max(8, c // r), c, 1), nn.Sigmoid())
        self.cv = Conv(c, c, 1, act=False)
        nn.init.zeros_(self.cv.bn.weight)  # the block starts as an identity

    def forward(self, x):
        y = torch.cat((self.pconv(x[:, :self.cp]), x[:, self.cp:]), 1)
        y = y * self.se(y)
        return x + self.cv(y)


class C3PD(nn.Module):
    """C3-PD: shallow branch (1x1 conv) + deep branch (1x1 conv -> PDBlocks), concat, 1x1 conv."""

    def __init__(self, c1, c2, n=1, e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 1)  # deep branch
        self.cv2 = Conv(c1, c_, 1)  # shallow branch
        self.m = nn.Sequential(*(PDBlock(c_) for _ in range(n)))
        self.cv3 = Conv(2 * c_, c2, 1)

    def forward(self, x):
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))


def _swap(model, i, new, name):
    old = model.model[i]
    new = new.to(next(model.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, name  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[i] = new


@torch.no_grad()
def add_cfa(model, layers=CFA_LAYERS):
    """Replace the Concat at the given neck layers with CFA. The fused channel count is read with one small forward pass."""
    chans, hooks = {}, []

    def record(i):
        def hook(m, inp, out):  # must return None, otherwise it would replace the layer output
            chans.setdefault(i, out.shape[1])
        return hook

    for i in layers:
        assert isinstance(model.model[i], Concat), f"layer {i} is {type(model.model[i]).__name__}, not Concat"
        hooks.append(model.model[i].register_forward_hook(record(i)))
    was_training = model.training
    model.eval()
    model(torch.zeros(1, 3, 64, 64, device=next(model.parameters()).device))
    model.train(was_training)
    for h in hooks:
        h.remove()
    for i in layers:
        _swap(model, i, CFA(chans[i], len(model.model[i].f)), "CFA")
    return model


def add_c3pd(model, layers=C3PD_LAYERS):
    """Replace the neck C3k2 at the given layers with C3-PD (same input and output channels, new weights)."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, C3k2), f"layer {i} is {type(old).__name__}, not C3k2"
        _swap(model, i, C3PD(old.cv1.conv.in_channels, old.cv2.conv.out_channels), "C3PD")
    return model


class CFATrainer(DetectionTrainer):
    """ADown YAML + pretrained weights + ECA (6, 8) + CFA at the neck fusion nodes."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_eca(model)
        add_cfa(model)
        if RANK in {-1, 0}:
            print(f"Backbone: ADown 3, 5, 7 + ECA after {list(ECA_LAYERS)} | Neck (Paper 22): CFA at {list(CFA_LAYERS)}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


class C3PDCFATrainer(DetectionTrainer):
    """ADown YAML + pretrained weights + ECA (6, 8) + CFA at the fusion nodes + C3-PD instead of the neck C3k2."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_eca(model)
        add_cfa(model)
        add_c3pd(model)
        if RANK in {-1, 0}:
            print(f"Backbone: ADown 3, 5, 7 + ECA | Neck (Paper 22): CFA at {list(CFA_LAYERS)} + C3-PD at {list(C3PD_LAYERS)}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


Writing cfa_neck.py


In [8]:
import importlib, eca_backbone, cfa_neck
importlib.reload(eca_backbone); importlib.reload(cfa_neck)
from eca_backbone import add_eca
from cfa_neck import add_cfa, add_c3pd, CFA, CFATrainer, C3PDCFATrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

MODEL_YAML = str(WORK / "yolo26n-final-backbone.yaml")

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs proposed backbone vs the two Paper 22 necks
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
prop = add_eca(DetectionModel(yaml_model_load(MODEL_YAML), nc=len(names), verbose=False)).eval()
m_cfa = add_cfa(deepcopy(prop)).eval()
m_both = add_c3pd(add_cfa(deepcopy(prop))).eval()
with torch.no_grad():
    for m in (m_cfa, m_both):
        _ = m(torch.zeros(1, 3, 640, 640))  # eval path
        _ = m(torch.zeros(1, 3, 640, 352))  # rect-style input, as in validation
        m.train(); _ = m(torch.zeros(2, 3, 640, 640)); m.eval()  # training path too
    # CFA: the training path (concat -> conv -> split) and the eval path (separate convs) must agree in eval mode
    cfa = CFA(32).eval()
    for p in cfa.parameters():
        p.data.normal_(0, 0.2)
    xs = [torch.randn(1, 16, 20, 24), torch.randn(1, 16, 20, 24)]
    ref = cfa(xs)
    cfa.training = True; cfa.bn1.training = False  # training branch, but BN with fixed statistics
    print(f"CFA eval path = paper path: max diff {(cfa(xs) - ref).abs().max():.2e} (should be about 1e-7)")
print("neck layers 12-22 (CFA):      ", [type(m_cfa.model[i]).__name__ for i in range(12, 23)])
print("neck layers 12-22 (C3-PD+CFA):", [type(m_both.model[i]).__name__ for i in range(12, 23)])
for tag, m in (("base YOLO26n", base), ("Proposed backbone", prop), ("+ CFA neck", m_cfa), ("+ C3-PD + CFA neck", m_both)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:20s} Params {pm:.3f} M | GFLOPs {gf:.2f}")
print("expected: + CFA a little above the proposed backbone (~2.03 M | ~4.49 GFLOPs); + C3-PD + CFA lower")

Overriding model.yaml nc=80 with nc=27
Overriding model.yaml nc=80 with nc=27
CFA eval path = paper path: max diff 0.00e+00 (should be about 1e-7)
neck layers 12-22 (CFA):       ['CFA', 'C3k2', 'Upsample', 'CFA', 'C3k2', 'Conv', 'CFA', 'C3k2', 'Conv', 'CFA', 'C3k2']
neck layers 12-22 (C3-PD+CFA): ['CFA', 'C3PD', 'Upsample', 'CFA', 'C3PD', 'Conv', 'CFA', 'C3PD', 'Conv', 'CFA', 'C3PD']
base YOLO26n         Params 2.380 M | GFLOPs 5.34
Proposed backbone    Params 2.034 M | GFLOPs 4.49
+ CFA neck           Params 2.075 M | GFLOPs 4.49
+ C3-PD + CFA neck   Params 1.703 M | GFLOPs 3.76
expected: + CFA a little above the proposed backbone (~2.03 M | ~4.49 GFLOPs); + C3-PD + CFA lower


## 5. Training

The two Paper 22 neck variants are trained one after another with identical settings (about 2–3 h each on a T4).
To skip a variant, put `#` in front of its line in `VARIANTS`.

**Kaggle 12 h limit:** if the run gets close to 12 hours, training stops by itself after the epoch that passes 10.5 h (counted from the setup cell) and the notebook continues with the evaluation, export and zip. The cell prints how many epochs were trained; write that number in the paper if it is below 100. A variant that has not started by then is skipped.

In [9]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name, trainer)
VARIANTS = [
    ("Proposed Backbone + CFA Neck (Paper 22)", "nk_cfa", CFATrainer),
    ("Proposed Backbone + C3-PD + CFA Neck (Paper 22)", "nk_c3pd_cfa", C3PDCFATrainer),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

# Kaggle stops a session after 12 hours. Training stops after the epoch that passes TRAIN_STOP_HOURS (counted from the
# setup cell), so the test evaluation, NCNN export and zip still finish in time. best.pt is the best epoch reached so far.
TRAIN_STOP_HOURS = 10.5

def stop_before_kaggle_limit(trainer):
    hours = (time.time() - NOTEBOOK_START) / 3600
    if hours > TRAIN_STOP_HOURS and not trainer.stop:
        trainer.stop = True
        print(f"\n⏱ {hours:.1f} h since the notebook started: training stops after epoch {trainer.epoch + 1} (Kaggle 12 h limit).")

for label, run, trainer in VARIANTS:
    if (time.time() - NOTEBOOK_START) / 3600 > TRAIN_STOP_HOURS:
        print(f"{run}: skipped, the 10.5 h guard has passed"); continue
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO(MODEL_YAML)
    model.add_callback("on_fit_epoch_end", stop_before_kaggle_limit)
    model.train(trainer=trainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    n_ep = len(pd.read_csv(Path(PROJECT) / f"{run}_s{SEED}" / "results.csv"))
    print(f"{run}: trained {n_ep} of {EPOCHS} epochs" + ("" if n_ep >= EPOCHS else " (stopped early: patience or the 12 h guard)"))
    del model; gc.collect(); torch.cuda.empty_cache()


========== Proposed Backbone + CFA Neck (Paper 22) (nk_cfa) ==========
New https://pypi.org/project/ultralytics/8.4.202 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.137 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/digitalaccess/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, li

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      4.24G        1.4      3.273    0.01922         14        640: 100% ━━━━━━━━━━━━ 482/482 4.0it/s 2:00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 6.3it/s 2.9s
                   all        550        557      0.664      0.654       0.76      0.407

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      4.25G      1.446      1.856    0.01977         12        640: 100% ━━━━━━━━━━━━ 482/482 4.1it/s 1:57
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 6.4it/s 2.8s
                   all        550        557      0.721      0.751      0.833       0.42

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      4.25G      1.435      1.484    0.01946          7        640: 100% ━━━━━━━━━━━━ 482/482 4.2it/s 1:56
                 Class     Images  Instances      Box(P 

## 6. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as every other row.

In [10]:
import eca_backbone, cfa_neck  # must be imported before loading the checkpoint

rows = []
for label, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

Ultralytics 8.4.137 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-final-backbone summary (fused): 147 layers, 2,074,909 parameters, 0 gradients, 4.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 876.7±162.6 MB/s, size: 37.0 KB)
val: Scanning /kaggle/working/digitalaccess/test/labels... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 1.1Kit/s 0.5s
val: New cache created: /kaggle/working/digitalaccess/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 12.0it/s 2.9s
                   all        550        552      0.968      0.936       0.95      0.584
Speed: 0.5ms preprocess, 3.0ms inference, 0.0ms loss, 0.2ms postprocess per image
Ultralytics 8.4.137 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n-final-backbone summary (fused): 151 layers, 1,702,525 parameters, 0 gradients, 3.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, r

,Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50
0,Proposed Backbone + CFA Neck (Paper 22),2.07,4.49,0.968,0.936,0.952,0.950
1,Proposed Backbone + C3-PD + CFA Neck (Paper 22),1.70,3.76,0.964,0.938,0.951,0.959


## 7. Package for Raspberry Pi 5 and Jetson

Creates `paper22_neck_edge_bench/` with:

| Item | Used on |
|---|---|
| `nk_cfa_ncnn_model/`, `nk_c3pd_cfa_ncnn_model/` (NCNN, imgsz 640, ECA as 1×1 conv) | Raspberry Pi 5 |
| `nk_cfa.pt`, `nk_c3pd_cfa.pt` + `eca_backbone.py` + `cfa_neck.py` | Jetson |
| `bench_images/` (the same 50 real test images, seed 0) | both |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Before the NCNN export, the 2 ECA blocks of the backbone are rewritten as exactly equivalent banded 1×1 convs (`fix_eca_ncnn.py`),
because the transpose-based ECA crashes NCNN. The `.pt` file for the Jetson keeps the original ECA.

In [11]:
import eca_backbone, cfa_neck  # needed to load the checkpoint
from fix_eca_ncnn import patch  # ECA -> banded 1x1 conv, NCNN-safe

EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper22_neck_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("eca_backbone.py", EDGE / "eca_backbone.py")
shutil.copy("cfa_neck.py", EDGE / "cfa_neck.py")

# the same 50 real test images as the earlier packages (same seed and selection)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    # NCNN-safe copy: ECA -> equivalent banded 1x1 conv (checked block by block), then export
    yolo = YOLO(str(w), task="detect")
    n, diff = patch(yolo.model.float().eval())
    print(f"{run}: replaced {n} ECA blocks for NCNN, max block output difference = {diff:.2e}")
    assert n == 2, "expected 2 ECA blocks (layers 6 and 8)"
    tmp = WORK / f"{run}_ncnn_safe.pt"
    yolo.save(tmp)
    p = YOLO(str(tmp), task="detect").export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import eca_backbone, cfa_neck  # needed to load the checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
print("\nEdge package ready. Run section 9 to zip everything.")

bench_images: 50 real test images
nk_cfa: replaced 2 ECA blocks for NCNN, max block output difference = 0.00e+00
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n-final-backbone summary (fused): 171 layers, 2,281,554 parameters, 0 gradients, 4.5 GFLOPs

PyTorch: starting from '/kaggle/working/nk_cfa_ncnn_safe.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 31, 8400) (4.8 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 1 package in 239ms
 Downloaded ncnn
Prepared 1 package in 228ms
Installed 1 package in 2ms
 + ncnn=

pnnxparam = /kaggle/working/nk_cfa_ncnn_safe_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/nk_cfa_ncnn_safe_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/nk_cfa_ncnn_safe_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/nk_cfa_ncnn_safe_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/nk_cfa_ncnn_safe_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/nk_cfa_ncnn_safe_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/nk_cfa_ncnn_safe_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = cfa_neck.CFA
inline module = eca_backbone.C3k2ECA
inline module = fix_eca_ncnn.ECAConv1x1
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.ADown
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.module

NCNN: export success ✅ 9.1s, saved as '/kaggle/working/nk_cfa_ncnn_safe_ncnn_model' (8.5 MB)

Export complete (9.8s)
Results saved to /kaggle/working/nk_cfa_ncnn_safe_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/nk_cfa_ncnn_safe_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/nk_cfa_ncnn_safe_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app
nk_cfa       -> nk_cfa.pt, nk_cfa_ncnn_model/
nk_c3pd_cfa: replaced 2 ECA blocks for NCNN, max block output difference = 0.00e+00
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
YOLO26n-final-backbone summary (fused): 175 layers, 1,909,170 parameters, 0 gradients, 3.8 GFLOPs

PyTorch: starting from '/kaggle/workin

pnnxparam = /kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = cfa_neck.C3PD
inline module = cfa_neck.CFA
inline module = cfa_neck.PDBlock
inline module = eca_backbone.C3k2ECA
inline module = fix_eca_ncnn.ECAConv1x1
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.

NCNN: export success ✅ 5.3s, saved as '/kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model' (7.1 MB)

Export complete (5.8s)
Results saved to /kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/nk_c3pd_cfa_ncnn_safe_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app
nk_c3pd_cfa  -> nk_c3pd_cfa.pt, nk_c3pd_cfa_ncnn_model/

Edge package ready. Run section 9 to zip everything.


### How to run on the Raspberry Pi 5

Copy `paper22_neck_all_results.zip` to the Pi and run the bench script:
```bash
~/fps/bench_paper.sh paper22_neck_all_results.zip
```

## 8. Neck table row

The reference row is the **proposed backbone with the YOLO26n neck** from `final_backbone.ipynb`. Type its values once they are known
(and the FPS of these models from the Pi 5), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 6 first in the same session (it fills `METRICS`).

In [12]:
# reference row: proposed backbone + YOLO26n neck (fill in from final_backbone.ipynb)
BASE_ROW = {"Variant": "Proposed Backbone + YOLO26n Neck", "Params (M)": 2.03, "GFLOPs": 4.49,
            "Precision": None, "Recall": None, "F1-score": None, "mAP@50": None,
            "FPS (Rasp-5)": None, "FPS (Jetson)": None}

FPS_PI5    = {"nk_cfa": None, "nk_c3pd_cfa": None}
FPS_JETSON = {"nk_cfa": None, "nk_c3pd_cfa": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50,FPS (Rasp-5),FPS (Jetson)
Proposed Backbone + YOLO26n Neck,2.030000,4.490000,--,--,--,--,--,--
Proposed Backbone + CFA Neck (Paper 22),2.070000,4.490000,0.968000,0.936000,0.952000,0.950000,--,--
Proposed Backbone + C3-PD + CFA Neck (Paper 22),1.700000,3.760000,0.964000,0.938000,0.951000,0.959000,--,--


## 9. Zip all results

Run this last. It creates **`paper22_neck_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper22_neck_edge_bench/` | NCNN + `.pt` model, module files, 50 bench images, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `nk_cfa_best.pt`, `nk_c3pd_cfa_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |
| `yolo26n-final-backbone.yaml`, `eca_backbone.py`, `cfa_neck.py` | the model definition |

In [13]:
import zipfile

ALL_ZIP = WORK / "paper22_neck_all_results.zip"
folders = [WORK / "paper22_neck_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run, _ in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")
    if Path(MODEL_YAML).exists():
        z.write(MODEL_YAML, Path(MODEL_YAML).name)
    z.write("eca_backbone.py", "eca_backbone.py")
    z.write("cfa_neck.py", "cfa_neck.py")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")

paper22_neck_edge_bench/       66 files
runs/                          45 files
weights/nk_cfa_best.pt
weights/nk_c3pd_cfa_best.pt

Download paper22_neck_all_results.zip (59.9 MB) from the Output tab.
